# Tercile Hit Rate Metric Heatmap Plots
This notebook walks through the generation of our Monthly Tercile Hit Rate Plots, assuming that you have monthly merged netCDF data that was generated with monthly_data_generation.py

In [3]:
# import necessary libraries
import pandas as pd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
import os

# set empty values in plots to grey
sns.set(rc={'axes.facecolor':'grey'})

# set png resolution
plt.rcParams['figure.dpi'] = 300

# Functions for data pre-processing

Code cell below defines helper functions for pre-processing the monthly merged netCDF data into a dataframe for the Tercile Hit Rate plots

In [5]:
def get_tercile_cutoffs(df):
    '''Helper function that takes a dataframe and returns the tercile cutoffs for each column
    '''
    return df.quantile([0.33, 0.66])

def assign_tercile_category(value, lower_cutoff, upper_cutoff):
    '''Helper function that takes a value and tercile cutoffs and returns the tercile category
    '''
    if value <= lower_cutoff:
        return 'Low'
    elif value <= upper_cutoff:
        return 'Medium'
    else:
        return 'High'

# generate a tercile dataframe for one region and one model
def compute_tercile_monthly_df(merged_file_path_nc):
  """This function takes a monthly merged netcdf file path and returns
  a dataframe with tercile classifications for predicted and observed
  precipitation, as well as the agreement between the two.

  Assumptions
  -----------
  The merged monthly netcdf file has the following variables, and was
  generated with monthly_data_generation.py:
  - predicted_precip: predicted precipitation
  - precip: observed precipitation
  - time: realization time
  - lead_time: lead time
  - latitude: latitude
  - longitude: longitude
  - M: ensemble member

  Returns
  -------
  A dataframe with the following columns:
  - year: year of the realization
  - lead_time: lead time
  - month: month of the realization
  - predicted_precip: predicted precipitation
  - precip: observed precipitation
  - {model}_tercile_class: tercile classification for predicted precipitation
  - chirps_tercile_class: tercile classification for observed precipitation
  - agreement: 1 if the tercile classifications agree, 0 otherwise
  - model: model name
  - region: region name

  Function Outline:
  1. Extract region and model name from the given file path
  2. Open netcdf file
  3. Take the ensemble mean
  4. Take the spatial means
  5. Subset data to 1993 through 2024
  6. Iterate over each unique combination of month and lead time, over all years
     a. Compute tercile cutoffs for predicted and observed precipitation
     b. Assign tercile categories for predicted and observed precipitation
     c. Calculate agreement (0/1)
  7. Return the dataframe for that model and that region of the netCDF file
  """

  # extract region and model name from netcdf path

  # Normalize the path to handle both slashes and backslashes
  normalized_path = os.path.normpath(merged_file_path_nc)

  # Extract just the filename
  filename = os.path.basename(normalized_path)

  # Split filename by underscores
  parts = filename.split('_')

  # Extract model and region
  model = parts[-2]
  region = '_'.join(parts[:-2])

  # status message
  print(f'Currently Processing {region} Region for {model } Model')

  # open netcdf file
  current_netcdf = xr.open_dataset(normalized_path)

  current_df = current_netcdf.to_dataframe()

  current_df = current_df.reset_index()

  # take the ensemble mean
  current_df = (current_df
                            .groupby(['time', 'lead_time', 'latitude', 'longitude'])[['predicted_precip', 'precip']]
                            .mean().reset_index())

  # take the spatial means
  current_df = (current_df  # Use the ensemble_means DataFrame
                              .groupby(['time', 'lead_time'])[['predicted_precip', 'precip']]
                              .mean().reset_index())

  # separate month and year into columns, drop time
  current_df['month'] = current_df['time'].dt.month
  current_df['year'] = current_df['time'].dt.year
  current_df = current_df.drop('time', axis=1, inplace=False)

  # reset index for wrangling
  current_df = current_df.reset_index().dropna()

  # subset to 1993 to 2024
  current_df = current_df.query('year >= 1993 and year <= 2024')

  # Initialize the `agreement` column
  current_df['agreement'] = None

  # Iterate over each unique combination of month and lead_time, over all years
  for (month, lead_time), group in current_df.groupby(['month', 'lead_time']):
      # Compute tercile cutoffs for predicted_precip and precip for this group (across all years)
      cutoffs = get_tercile_cutoffs(group[['predicted_precip', 'precip']])

      # Extract cutoffs for predicted_precip and precip separately
      lower_cutoff_predicted = cutoffs.loc[0.33, 'predicted_precip']
      upper_cutoff_predicted = cutoffs.loc[0.66, 'predicted_precip']

      lower_cutoff_precip = cutoffs.loc[0.33, 'precip']
      upper_cutoff_precip = cutoffs.loc[0.66, 'precip']

      # assign tercile categories for predicted_precip and precip
      group[f'{model}_tercile_class'] = group['predicted_precip'].apply(assign_tercile_category, args=(lower_cutoff_predicted, upper_cutoff_predicted))
      group['chirps_tercile_class'] = group['precip'].apply(assign_tercile_category, args=(lower_cutoff_precip, upper_cutoff_precip))

      # Calculate agreement: 1 if terciles agree, 0 otherwise
      group['agreement'] = (group[f'{model}_tercile_class'] == group['chirps_tercile_class']).astype(int)

      # Update the DataFrame with the new columns
      current_df.loc[group.index, f'{model}_tercile_class'] = group[f'{model}_tercile_class']
      current_df.loc[group.index, 'chirps_tercile_class'] = group['chirps_tercile_class']
      current_df.loc[group.index, 'agreement'] = group['agreement']
      current_df.loc[group.index, 'model'] = model
      current_df.loc[group.index, 'region'] = region

  # Select the desired columns for the final DataFrame
  final_columns = ['year', 'lead_time', 'month', 'predicted_precip', 'precip', f'{model}_tercile_class', 'chirps_tercile_class', 'agreement', 'model', 'region']
  final_df = current_df[final_columns]

  return final_df

# Example Usage of the Functions to generate Tercile Hit Rate Plots
- Iterate over all merged monthly netCDF files in a folder and generate plotting dataframes
- Subset combined merged plotting dataframe into separate AN/N/BN Tercile plotting dataframes
- Generate AN/N/BN Tercile Hit Rate plots

In [ ]:
'''Generate plotting dataframes using all merged monthly netCDF files
'''

# Initialize an empty dictionary to store DataFrames
dfs_dict = {}

# Define the monthly merged netCDF directory path
dir_path = '/content/drive/MyDrive/capstone_data/netCDF/'

# Get list of all full file paths
list_of_paths = [os.path.join(dir_path, f) for f in os.listdir(dir_path)]

# Loop over all files
for fpath in list_of_paths:
    # Generate the tercile plotting DataFrame for the current file
    df = compute_tercile_monthly_df(fpath)

    # Store the DataFrame in the dictionary with the path as key
    dfs_dict[fpath] = df

# Concatenate all DataFrames in the dictionary into one DataFrame
final_df = pd.concat(dfs_dict.values(), ignore_index=True)

# now, 'final_df' contains tercile hit rate information for all the regions
# and models that were in the monthly merged netCDF directory

In [8]:
'''Subset the final combined dataframe by tercile class, low, medium, and high for plotting
'''

# subset by chirps tercile class = High, compute agreement rates
AN_tercile_df = final_df.query('chirps_tercile_class == "High"').groupby(['month', 'lead_time', 'model', 'region'])[['agreement']].mean().reset_index()

# subset by chirps tercile class = medium, compute agreement rates
N_tercile_df = final_df.query('chirps_tercile_class == "Medium"').groupby(['month', 'lead_time', 'model', 'region'])[['agreement']].mean().reset_index()

# subset by chirps tercile class = low, compute agreement rates
BN_tercile_df = final_df.query('chirps_tercile_class == "Low"').groupby(['month', 'lead_time', 'model', 'region'])[['agreement']].mean().reset_index()

In [16]:
'''Helper Function for Plotting Tercile Hit Rate'''

def draw_heatmap(*args, **kwargs):
    """Draws a heatmap for Tercile Hit Rate using the calculated agreement

       Note:
       Values bounded from 0 to 1, using Red/Yellow/green colors for plotting
    """
    # access the dataframe that is passed into the function
    data = kwargs.pop('data')

    # pivot data, index is month, columns is lead time, values is agreement rate
    d = data.pivot(index=args[1], columns=args[0], values=args[2])

    # Convert the data to numeric, handling errors by coercing to NaN
    d = d.apply(pd.to_numeric, errors='coerce')

    # define a heatmap with pivoted data, value bounds, and colors
    sns.heatmap(d, **kwargs, vmin=0, vmax=1,
                cmap=sns.color_palette('RdYlGn', 12),
                linewidths=0.1, linecolor='black')

    # heatmap labels
    plt.xticks(np.arange(0.5, 12.5, 1))  # Set lead time ticks explicitly
    plt.gca().set_xticklabels(np.arange(0.5, 12.5, 1))  # Set lead time labels
    plt.xticks(fontsize=6)
    plt.yticks(fontsize=6)
    plt.xticks(rotation=0) # set X-axis labels to 0 rotation
    plt.gca().invert_yaxis() # invert y axis; 1 appears at the bottom

In [17]:
'''Plot AN Tercile Hit Rate for all models and all regions
'''

# Set large facetGrid, where columns is region and rows are models
fg = sns.FacetGrid(AN_tercile_df, col='region', row='model', sharex=False, sharey=False)

# call helper function to draw heatmap onto the facet grid
fg.map_dataframe(draw_heatmap, 'lead_time', 'month', 'agreement', square = True)

# set main and axis titles
fg.set_titles('AN Tercile Hit Rate \n Region: {col_name} \n Model: {row_name}')
fg.set_ylabels("Month")
fg.set_xlabels("Lead Time")

# save figure
plt.savefig('figures/AN_Tercile_Hit_Rate_Monthly.png')

# close for efficiency reasons
plt.close()

In [40]:
'''Plot N Tercile Hit Rate for all models and all regions
'''

# Set large facetGrid, where columns is region and rows are models
fg = sns.FacetGrid(N_tercile_df, col='region', row='model', sharex=False, sharey=False)

# call helper function to draw heatmap onto the facet grid
fg.map_dataframe(draw_heatmap, 'lead_time', 'month', 'agreement', square = True)

# set main and axis titles
fg.set_titles('N Tercile Hit Rate \n Region: {col_name} \n Model: {row_name}')
fg.set_ylabels("Month")
fg.set_xlabels("Lead Time")

# save figure
plt.savefig('figures/N_Tercile_Hit_Rate_Monthly.png')

# close for efficiency reasons
plt.close()

In [37]:
'''Plot BN Tercile Hit Rate for all models and all regions
'''

# Set large facetGrid, where columns is region and rows are models
fg = sns.FacetGrid(BN_tercile_df, col='region', row='model', sharex=False, sharey=False)

# call helper function to draw heatmap onto the facet grid
fg.map_dataframe(draw_heatmap, 'lead_time', 'month', 'agreement', square = True)

# set main and axis titles
fg.set_titles('BN Tercile Hit Rate \n Region: {col_name} \n Model: {row_name}')
fg.set_ylabels("Month")
fg.set_xlabels("Lead Time")

# save figure
plt.savefig('figures/BN_Tercile_Hit_Rate_Monthly.png')

# close for efficiency reasons
plt.close()